# Lab 2 Parallel Computing

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Lab-2-2-ParallelProcessing") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.enabled", "true") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "lab_2/lakehouse") 

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark version: {spark.version} with Delta support Lab-2")

Spark version: 3.5.0 with Delta support Lab-2


## 1. Distributed data generation

In [2]:
from faker import Faker
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import random

# 1. Create "empty" DataFrame with huge amount of rows
# 100,000 to start with so as not to overload your PC's RAM
print("Processing Start ......")
num_rows = 300000
df = spark.range(0, num_rows).repartition(4) # split by 4 parts (partitions)

# 2. Using UDFs to create data in Spark 
fake = Faker()
def get_random_city():
    return random.choice(["Kyiv", "Lviv", "Odesa", "Dnipro", "Kharkiv"])

city_udf = udf(get_random_city, StringType())

# Add columns
df_large = df.withColumn("city", city_udf()) \
             .withColumn("amount", (df.id * 0.01).cast("decimal(18,2)"))

print("Processing Stop")
df_large.show(5)

Processing Start ......
Processing Stop
+-----+-------+------+
|   id|   city|amount|
+-----+-------+------+
| 5401|  Odesa| 54.01|
| 9782|Kharkiv| 97.82|
|13249|   Kyiv|132.49|
| 1844|   Lviv| 18.44|
|23427|Kharkiv|234.27|
+-----+-------+------+
only showing top 5 rows



## 2. Partitioning by City

This is analogous to partitioning in Oracle. We will write the data so that each city is in a separate folder. This allows Spark to read only the data it needs when you do WHERE city = 'Kyiv'.

In [3]:
# write with Partitioning
print("Processing Start ......")
df_large.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("city") \
    .saveAsTable("partitioned_transactions")

print("Processing Stop")

Processing Start ......
Processing Stop
